In [3]:
from pathlib import Path
import io
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Đặt file này trực tiếp trong thư mục measurement-scripts.
# Script tự đọc dữ liệu từ ./kb5 và ./kb6.

try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

KB5_DIR = BASE_DIR / "kb5"
KB6_DIR = BASE_DIR / "kb6"
RESULTS_DIR = BASE_DIR / "results"
CHART_DIR = BASE_DIR / "ucon_measurement_analysis_postupdate" / "charts"
DATA_DIR = BASE_DIR / "ucon_measurement_analysis_postupdate" / "data"
CHART_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

K_ORDER = [1, 10, 50, 100]
N_ORDER = [100, 500, 1000]

CORE_COLORS = {
    "dependency_lookup_ms": "#159A8C",
    "pip_fetch_ms": "#2F66E4",
    "pdp_evaluate_ms": "#7C3AED",
    "revoke_update_ms": "#F59E0B",
    "post_processing_ms": "#E12726",
    "revoke_publish_ms": "#0E7490",
    "unattributed_core_ms": "#A05A2C",
}

OUTSIDE = {
    "poller": "#D9D9D9",
    "residual": "#9E9E9E",
}

COMPONENTS = [
    "dependency_lookup_ms",
    "pip_fetch_ms",
    "pdp_evaluate_ms",
    "revoke_update_ms",
    "post_processing_ms",
    "revoke_publish_ms",
    "unattributed_core_ms",
]

LABELS = {
    "dependency_lookup_ms": "Dependency Lookup",
    "pip_fetch_ms": "PIP",
    "pdp_evaluate_ms": "PDP",
    "revoke_update_ms": "Revoke Update",
    "post_processing_ms": "POST Processing",
    "revoke_publish_ms": "Revoke Publish",
    "unattributed_core_ms": "Chi phí chưa phân rã",
}

plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"


def clean_csv(raw_bytes: bytes, scenario: str) -> pd.DataFrame:
    raw = raw_bytes.decode("utf-8")
    raw = raw.replace("\r\n", "\n").replace("\r", "\n")
    raw = re.sub(rf"(?<!\n)({re.escape(scenario)},)", r"\n\1", raw)
    return pd.read_csv(io.StringIO(raw))


def load_kb5() -> pd.DataFrame:
    files = sorted(KB5_DIR.glob("core_breakdown_summary_*.csv"))
    if len(files) != 8:
        raise FileNotFoundError(
            f"KB5 cần 8 summary CSV, hiện tìm thấy {len(files)} file trong {KB5_DIR}"
        )

    frames = []
    for path in files:
        df = clean_csv(path.read_bytes(), "core_breakdown")
        frames.append(df)

    data = pd.concat(frames, ignore_index=True)

    numeric = [
        "requested_session_count", "affected_session_count",
        "poller_wait_ms", "event_processing_latency_ms",
        "event_end_to_end_latency_ms", "dependency_lookup_ms",
        "pip_fetch_ms", "pdp_evaluate_ms", "revoke_update_ms",
        "post_processing_ms", "revoke_publish_ms",
        "measured_component_sum_ms", "unattributed_core_ms",
        "reevaluation_total_ms", "total_core_ms",
    ]
    for c in numeric:
        data[c] = pd.to_numeric(data[c], errors="coerce")

    data = data[
        data["success"].astype(str).str.lower().eq("true")
        & data["requested_session_count"].isin(K_ORDER)
    ].copy()

    counts = data.groupby("requested_session_count").size().reindex(K_ORDER)
    if not counts.eq(100).all():
        raise ValueError(f"KB5 không đủ 100 lần đo/K:\n{counts}")

    if not (data["affected_session_count"] == data["requested_session_count"]).all():
        raise ValueError("KB5 có affected_session_count không khớp K.")

    if not np.allclose(
        data["event_end_to_end_latency_ms"],
        data["poller_wait_ms"] + data["event_processing_latency_ms"],
        atol=0.002,
    ):
        raise ValueError("KB5: công thức E2E không khớp.")

    if not np.allclose(
        data["total_core_ms"],
        data["measured_component_sum_ms"] + data["unattributed_core_ms"],
        atol=0.002,
    ):
        raise ValueError("KB5: công thức total_core_ms không khớp.")

    return data


def load_kb6() -> pd.DataFrame:
    # Trường hợp 1: summary CSV nằm trực tiếp trong kb6 hoặc thư mục con.
    files = sorted(KB6_DIR.rglob("scalability_summary_*.csv"))

    frames = []

    for path in files:
        frames.append(clean_csv(path.read_bytes(), "scalability"))

    # Trường hợp 2: dữ liệu được nén trong scalability.zip.
    if not frames:
        zip_candidates = list(KB6_DIR.rglob("scalability.zip"))

        for zip_path in zip_candidates:
            with zipfile.ZipFile(zip_path) as z:
                for name in sorted(z.namelist()):
                    if "scalability_summary_" in name and name.endswith(".csv"):
                        frames.append(clean_csv(z.read(name), "scalability"))

    if not frames:
        raise FileNotFoundError(
            f"Không tìm thấy scalability_summary_*.csv hoặc scalability.zip trong {KB6_DIR}"
        )

    data = pd.concat(frames, ignore_index=True)

    numeric = [
        "total_active_sessions", "background_active_sessions",
        "requested_affected_session_count",
        "measured_affected_session_count",
        "active_sessions_before_event", "active_sessions_after_event",
        "poller_wait_ms", "event_processing_latency_ms",
        "event_end_to_end_latency_ms", "dependency_lookup_ms",
        "pip_fetch_ms", "pdp_evaluate_ms", "revoke_update_ms",
        "post_processing_ms", "revoke_publish_ms",
        "total_core_ms", "measured_component_sum_ms",
        "unattributed_core_ms", "reevaluation_total_ms",
        "last_revoke_latency_ms",
    ]

    for c in numeric:
        data[c] = pd.to_numeric(data[c], errors="coerce")

    data = data[
        data["success"].astype(str).str.lower().eq("true")
        & data["total_active_sessions"].isin(N_ORDER)
    ].copy()

    counts = data.groupby("total_active_sessions").size().reindex(N_ORDER)
    if not counts.eq(100).all():
        raise ValueError(f"KB6 không đủ 100 lần đo/N:\n{counts}")

    if not (data["requested_affected_session_count"] == 10).all():
        raise ValueError("KB6: K yêu cầu không bằng 10.")

    if not (data["measured_affected_session_count"] == 10).all():
        raise ValueError("KB6: số phiên thực tế bị ảnh hưởng không bằng 10.")

    if not (
        data["active_sessions_before_event"] == data["total_active_sessions"]
    ).all():
        raise ValueError("KB6: số phiên ACTIVE trước event không khớp N.")

    if not (
        data["active_sessions_after_event"] == data["total_active_sessions"] - 10
    ).all():
        raise ValueError("KB6: số phiên ACTIVE sau event không khớp N - 10.")

    return data


def save(fig, filename):
    fig.tight_layout()
    fig.savefig(
        CHART_DIR / filename,
        dpi=300,
        bbox_inches="tight",
        facecolor="white",
    )
    plt.close(fig)


kb5 = load_kb5()
kb6 = load_kb6()

# ============================================================
# Tổng hợp KB5
# ============================================================

kb5_agg = kb5.groupby("requested_session_count").agg(
    poller_wait_mean=("poller_wait_ms", "mean"),
    event_processing_mean=("event_processing_latency_ms", "mean"),
    e2e_mean=("event_end_to_end_latency_ms", "mean"),
    total_core_mean=("total_core_ms", "mean"),
    total_core_p95=("total_core_ms", lambda s: s.quantile(0.95)),
).reindex(K_ORDER)

for c in COMPONENTS:
    kb5_agg[c] = (
        kb5.groupby("requested_session_count")[c]
        .mean()
        .reindex(K_ORDER)
    )

kb5_agg["processing_residual_mean"] = (
    kb5_agg["event_processing_mean"] - kb5_agg["total_core_mean"]
)

labels_k = [f"K = {k}" for k in K_ORDER]

# Hình 13
fig, ax = plt.subplots(figsize=(11.8, 6.8))
bottom = np.zeros(len(K_ORDER))

vals = kb5_agg["poller_wait_mean"].to_numpy()
ax.bar(
    labels_k, vals, bottom=bottom,
    color=OUTSIDE["poller"], edgecolor="white",
    linewidth=0.8, label="Poller Wait"
)
bottom += vals

for c in COMPONENTS:
    vals = kb5_agg[c].to_numpy()
    ax.bar(
        labels_k, vals, bottom=bottom,
        color=CORE_COLORS[c], edgecolor="white",
        linewidth=0.8, label=LABELS[c]
    )
    bottom += vals

vals = kb5_agg["processing_residual_mean"].to_numpy()
ax.bar(
    labels_k, vals, bottom=bottom,
    color=OUTSIDE["residual"], edgecolor="white",
    linewidth=0.8, label="Phần xử lý ngoài Core"
)

for i, k in enumerate(K_ORDER):
    y = kb5_agg.loc[k, "e2e_mean"]
    ax.text(i, y + max(8, y * 0.012), f"{y:.0f} ms",
            ha="center", fontsize=10, )

ax.set_title(
    "Phân rã độ trễ đầu-cuối của event theo số phiên bị ảnh hưởng",
    fontsize=16, 
)
ax.set_xlabel("Số phiên bị ảnh hưởng K")
ax.set_ylabel("Thời gian trung bình (ms)")
ax.grid(axis="y", alpha=0.22)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
save(fig, "13_kb5_e2e_breakdown_with_poller.png")

# Hình 14
fig, ax = plt.subplots(figsize=(11.8, 6.8))
bottom = np.zeros(len(K_ORDER))

for c in COMPONENTS:
    vals = kb5_agg[c].to_numpy()
    ax.bar(
        labels_k, vals, bottom=bottom,
        color=CORE_COLORS[c], edgecolor="white",
        linewidth=0.8, label=LABELS[c]
    )
    bottom += vals

for i, k in enumerate(K_ORDER):
    y = kb5_agg.loc[k, "total_core_mean"]
    ax.text(i, y + max(5, y * 0.015), f"{y:.0f} ms",
            ha="center", fontsize=10, )

ax.set_title(
    "Phân rã thời gian xử lý bên trong UCON Core",
    fontsize=16, 
)
ax.set_xlabel("Số phiên bị ảnh hưởng K")
ax.set_ylabel("Thời gian trung bình (ms)")
ax.grid(axis="y", alpha=0.22)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
save(fig, "14_kb5_core_breakdown.png")

# Hình 15
fig, ax = plt.subplots(figsize=(11.8, 6.8))
bottom = np.zeros(len(K_ORDER))

for c in COMPONENTS:
    vals = 100 * kb5_agg[c].to_numpy() / kb5_agg["total_core_mean"].to_numpy()
    ax.bar(
        labels_k, vals, bottom=bottom,
        color=CORE_COLORS[c], edgecolor="white",
        linewidth=0.8, label=LABELS[c]
    )
    bottom += vals

ax.set_title(
    "Tỷ trọng các thành phần trong thời gian xử lý UCON Core",
    fontsize=16, 
)
ax.set_xlabel("Số phiên bị ảnh hưởng K")
ax.set_ylabel("Tỷ trọng trong total_core_ms (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.22)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=9)
save(fig, "15_kb5_core_component_share.png")



# Lưu toàn bộ bảng tổng hợp KB5 dùng để vẽ hình và viết phân tích.
kb5_agg.to_csv(
    DATA_DIR / "kb5_aggregated.csv",
    encoding="utf-8-sig",
)

# ============================================================
# Tổng hợp KB6
# ============================================================

kb6_agg = kb6.groupby("total_active_sessions").agg(
    dependency_lookup_mean=("dependency_lookup_ms", "mean"),
    dependency_lookup_p95=("dependency_lookup_ms", lambda s: s.quantile(0.95)),
    total_core_mean=("total_core_ms", "mean"),
    total_core_p95=("total_core_ms", lambda s: s.quantile(0.95)),
).reindex(N_ORDER)

x = np.arange(len(N_ORDER))
labels_n = [f"N = {n}" for n in N_ORDER]

# Hình 16
fig, ax = plt.subplots(figsize=(10.6, 6.3))

means = kb6_agg["dependency_lookup_mean"].to_numpy()
p95s = kb6_agg["dependency_lookup_p95"].to_numpy()

ax.bar(
    x, means, width=0.52, color="#159A8C",
    edgecolor="white", linewidth=0.8, label="Trung bình"
)
ax.plot(
    x, p95s, marker="o", linestyle="--",
    linewidth=2.1, color="#E12726", label="P95"
)

for i, v in enumerate(means):
    ax.text(i, v + 0.035, f"{v:.2f}",
            ha="center", fontsize=10, )

for i, v in enumerate(p95s):
    ax.text(i, v + 0.035, f"{v:.2f}",
            ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels_n)
ax.set_title(
    "Thời gian xác định phiên bị ảnh hưởng khi tổng số phiên tăng",
    fontsize=16, 
)
ax.set_xlabel("Tổng số phiên ACTIVE trong hệ thống N (K = 10 cố định)")
ax.set_ylabel("dependency_lookup_ms")
ax.set_ylim(0, 1.25)
ax.grid(axis="y", alpha=0.22)
ax.legend()
save(fig, "16_kb6_dependency_lookup.png")

# Hình 17
fig, ax = plt.subplots(figsize=(10.6, 6.3))

means = kb6_agg["total_core_mean"].to_numpy()
p95s = kb6_agg["total_core_p95"].to_numpy()

ax.bar(
    x, means, width=0.52, color="#2F66E4",
    edgecolor="white", linewidth=0.8, label="Trung bình"
)
ax.plot(
    x, p95s, marker="o", linestyle="--",
    linewidth=2.1, color="#E12726", label="P95"
)

for i, v in enumerate(means):
    ax.text(i, v + 1.5, f"{v:.1f}",
            ha="center", fontsize=10, )

for i, v in enumerate(p95s):
    ax.text(i, v + 1.5, f"{v:.1f}",
            ha="center", fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(labels_n)
ax.set_title(
    "Thời gian xử lý UCON Core khi tổng số phiên tăng",
    fontsize=16, 
)
ax.set_xlabel("Tổng số phiên ACTIVE trong hệ thống N (K = 10 cố định)")
ax.set_ylabel("total_core_ms")
ax.set_ylim(0, 130)
ax.grid(axis="y", alpha=0.22)
ax.legend()
save(fig, "17_kb6_total_core.png")

print("Đã tạo xong 5 biểu đồ:")
for name in [
    "13_kb5_e2e_breakdown_with_poller.png",
    "14_kb5_core_breakdown.png",
    "15_kb5_core_component_share.png",
    "16_kb6_dependency_lookup.png",
    "17_kb6_total_core.png",
]:
    print(" -", CHART_DIR / name)




# Lưu toàn bộ bảng tổng hợp KB6 dùng để vẽ hình và viết phân tích.
kb6_agg.to_csv(
    DATA_DIR / "kb6_aggregated.csv",
    encoding="utf-8-sig",
)

# ============================================================
# ĐÁNH GIÁ TÍNH KỊP THỜI THEO NGƯỠNG 1 GIÂY
#
# Dữ liệu sử dụng:
# - results/db_event_baseline_2.zip
# - results/db_event_multi_session_2.zip
# - results/timer_expiry_2.zip
#
# Ngưỡng 1000 ms là ngưỡng tham chiếu ở mức ứng dụng.
# Một cấu hình được xem là đáp ứng theo tiêu chí P95 khi
# ít nhất 95% số mẫu có thời gian thu hồi <= 1000 ms.
#
# Mốc 150 ms chỉ dùng để tham khảo thêm, KHÔNG dùng
# làm tiêu chí đạt/không đạt chính vì thuộc một hệ thống
# Zero Trust IoT khác.
# ============================================================

MAIN_THRESHOLD_MS = 1000
STRICT_REFERENCE_MS = 150
PASS_RATE_REQUIRED = 95.0


def load_result_zip(zip_path: Path) -> pd.DataFrame:
    if not zip_path.exists():
        raise FileNotFoundError(f"Không tìm thấy dữ liệu: {zip_path}")

    frames = []

    with zipfile.ZipFile(zip_path) as z:
        for name in sorted(z.namelist()):
            if name.endswith(".csv"):
                df = pd.read_csv(io.BytesIO(z.read(name)))
                df["source_file"] = name
                frames.append(df)

    if not frames:
        raise ValueError(f"Không có CSV trong {zip_path}")

    return pd.concat(frames, ignore_index=True)


def calculate_threshold_results():
    baseline = load_result_zip(
        RESULTS_DIR / "db_event_baseline_2.zip"
    )

    multi = load_result_zip(
        RESULTS_DIR / "db_event_multi_session_2.zip"
    )

    timer = load_result_zip(
        RESULTS_DIR / "timer_expiry_2.zip"
    )

    multi_summary = multi[
        multi["session_sequence"].isna()
    ].copy()

    multi_detail = multi[
        multi["session_sequence"].notna()
    ].copy()

    event_rows = []

    for poller, group in baseline.groupby("poller_interval_ms"):
        count = len(group)

        passed = (
            group["revoke_latency_ms"]
            <= MAIN_THRESHOLD_MS
        ).sum()

        strict_passed = (
            group["revoke_latency_ms"]
            <= STRICT_REFERENCE_MS
        ).sum()

        event_rows.append({
            "poller_interval_ms": int(poller),
            "samples": count,
            "p95_ms": group["revoke_latency_ms"].quantile(0.95),
            "passed_1000ms": int(passed),
            "pass_rate_1000ms_pct": 100 * passed / count,
            "pass_rate_150ms_pct": 100 * strict_passed / count,
        })

    multi_rows = []

    for k, group in multi_summary.groupby("requested_session_count"):
        count = len(group)

        # Tiêu chí event:
        # phiên bị thu hồi muộn nhất cũng phải <= 1 giây.
        passed_events = (
            group["revoke_max_ms"]
            <= MAIN_THRESHOLD_MS
        ).sum()

        details = multi_detail[
            multi_detail["requested_session_count"] == k
        ]

        passed_sessions = (
            details["revoke_latency_ms"]
            <= MAIN_THRESHOLD_MS
        ).sum()

        multi_rows.append({
            "requested_session_count": int(k),
            "events": count,
            "passed_events_1000ms": int(passed_events),
            "event_pass_rate_1000ms_pct":
                100 * passed_events / count,
            "session_pass_rate_1000ms_pct":
                100 * passed_sessions / len(details),
        })

    timer_rows = []

    for scheduler, group in timer.groupby("timer_scheduler_interval_ms"):
        count = len(group)

        passed = (
            group["timer_total_revoke_latency_ms"]
            <= MAIN_THRESHOLD_MS
        ).sum()

        strict_passed = (
            group["timer_total_revoke_latency_ms"]
            <= STRICT_REFERENCE_MS
        ).sum()

        timer_rows.append({
            "scheduler_interval_ms": int(scheduler),
            "samples": count,
            "p95_ms":
                group["timer_total_revoke_latency_ms"].quantile(0.95),
            "passed_1000ms": int(passed),
            "pass_rate_1000ms_pct": 100 * passed / count,
            "pass_rate_150ms_pct": 100 * strict_passed / count,
        })

    return (
        pd.DataFrame(event_rows)
        .sort_values("poller_interval_ms"),
        pd.DataFrame(multi_rows)
        .sort_values("requested_session_count"),
        pd.DataFrame(timer_rows)
        .sort_values("scheduler_interval_ms"),
    )


event_threshold, multi_threshold, timer_threshold = (
    calculate_threshold_results()
)

print()
print("=" * 72)
print("ĐÁNH GIÁ NGƯỠNG 1 GIÂY")
print("=" * 72)

print("\nEvent-driven, một phiên:")
print(event_threshold.round(3))

print("\nEvent-driven, nhiều phiên:")
print(multi_threshold.round(3))

print("\nTimer-based:")
print(timer_threshold.round(3))


# ============================================================
# HÌNH 18 — Event-driven một phiên
# ============================================================

fig, ax = plt.subplots(figsize=(10.8, 6.2))

labels = [
    f"{v} ms"
    for v in event_threshold["poller_interval_ms"]
]

rates = event_threshold[
    "pass_rate_1000ms_pct"
].to_numpy()

bars = ax.bar(
    labels,
    rates,
    color="#2F66E4",
    edgecolor="white",
    linewidth=0.8,
)

ax.axhline(
    PASS_RATE_REQUIRED,
    color="#E12726",
    linestyle="--",
    linewidth=2,
    label="Mức yêu cầu: ≥ 95% mẫu",
)

for bar, value in zip(bars, rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 2,
        f"{value:.0f}%",
        ha="center",
        
    )

ax.set_ylim(0, 105)
ax.set_title(
    "Tỷ lệ thu hồi trong 1 giây theo chu kỳ poller",
    fontsize=16,
    
)
ax.set_xlabel("Chu kỳ poller")
ax.set_ylabel(
    "Tỷ lệ mẫu có revoke_latency_ms ≤ 1000 ms (%)"
)
ax.grid(axis="y", alpha=0.22)
ax.legend()

save(
    fig,
    "18_timeliness_threshold_event_driven.png",
)


# ============================================================
# HÌNH 19 — Nhiều phiên
# ============================================================

ks = multi_threshold[
    "requested_session_count"
].tolist()

event_rates = multi_threshold[
    "event_pass_rate_1000ms_pct"
].to_numpy()

session_rates = multi_threshold[
    "session_pass_rate_1000ms_pct"
].to_numpy()

x = np.arange(len(ks))
width = 0.34

fig, ax = plt.subplots(figsize=(10.8, 6.2))

bars1 = ax.bar(
    x - width / 2,
    event_rates,
    width,
    color="#159A8C",
    edgecolor="white",
    linewidth=0.8,
    label="Event: tất cả phiên ≤ 1 giây",
)

bars2 = ax.bar(
    x + width / 2,
    session_rates,
    width,
    color="#2F66E4",
    edgecolor="white",
    linewidth=0.8,
    label="Từng phiên ≤ 1 giây",
)

ax.axhline(
    PASS_RATE_REQUIRED,
    color="#E12726",
    linestyle="--",
    linewidth=2,
    label="Mức yêu cầu: ≥ 95%",
)

for bars, values in [
    (bars1, event_rates),
    (bars2, session_rates),
]:
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 1.5,
            f"{value:.1f}%",
            ha="center",
            fontsize=9,
            
        )

ax.set_xticks(x)
ax.set_xticklabels([f"K = {k}" for k in ks])
ax.set_ylim(0, 107)

ax.set_title(
    "Tỷ lệ thu hồi trong 1 giây khi một event ảnh hưởng nhiều phiên",
    fontsize=16,
    
)
ax.set_xlabel("Số phiên bị ảnh hưởng K")
ax.set_ylabel("Tỷ lệ đáp ứng ngưỡng 1 giây (%)")
ax.grid(axis="y", alpha=0.22)
ax.legend()

save(
    fig,
    "19_timeliness_threshold_multi_session.png",
)


# ============================================================
# HÌNH 20 — Timer-based
# ============================================================

fig, ax = plt.subplots(figsize=(10.8, 6.2))

labels = [
    f"{v} ms"
    for v in timer_threshold["scheduler_interval_ms"]
]

rates = timer_threshold[
    "pass_rate_1000ms_pct"
].to_numpy()

bars = ax.bar(
    labels,
    rates,
    color="#7C3AED",
    edgecolor="white",
    linewidth=0.8,
)

ax.axhline(
    PASS_RATE_REQUIRED,
    color="#E12726",
    linestyle="--",
    linewidth=2,
    label="Mức yêu cầu: ≥ 95% mẫu",
)

for bar, value in zip(bars, rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 2,
        f"{value:.0f}%",
        ha="center",
        
    )

ax.set_ylim(0, 105)

ax.set_title(
    "Tỷ lệ thu hồi trong 1 giây theo chu kỳ scheduler",
    fontsize=16,
    
)
ax.set_xlabel(
    "Chu kỳ scheduler (poller cố định 1000 ms)"
)
ax.set_ylabel(
    "Tỷ lệ mẫu có timer_total_revoke_latency_ms ≤ 1000 ms (%)"
)
ax.grid(axis="y", alpha=0.22)
ax.legend()

save(
    fig,
    "20_timeliness_threshold_timer_based.png",
)


# ============================================================
# XUẤT TOÀN BỘ SỐ LIỆU DÙNG CHO PHẦN PHÂN TÍCH THRESHOLD
# ============================================================

event_threshold.to_csv(
    DATA_DIR / "threshold_event_driven.csv",
    index=False,
    encoding="utf-8-sig",
)

multi_threshold.to_csv(
    DATA_DIR / "threshold_multi_session.csv",
    index=False,
    encoding="utf-8-sig",
)

timer_threshold.to_csv(
    DATA_DIR / "threshold_timer_based.csv",
    index=False,
    encoding="utf-8-sig",
)

threshold_rows = []

for _, r in event_threshold.iterrows():
    threshold_rows.append({
        "scenario": "event_driven_single",
        "setting": f"poller={int(r['poller_interval_ms'])}",
        "samples": int(r["samples"]),
        "p95_ms": r["p95_ms"],
        "passed_1000ms": int(r["passed_1000ms"]),
        "pass_rate_1000ms_pct": r["pass_rate_1000ms_pct"],
        "pass_rate_150ms_pct": r["pass_rate_150ms_pct"],
        "meets_95pct_requirement":
            r["pass_rate_1000ms_pct"] >= PASS_RATE_REQUIRED,
    })

for _, r in multi_threshold.iterrows():
    threshold_rows.append({
        "scenario": "event_driven_multi",
        "setting": f"K={int(r['requested_session_count'])}",
        "samples": int(r["events"]),
        "passed_1000ms": int(r["passed_events_1000ms"]),
        "pass_rate_1000ms_pct": r["event_pass_rate_1000ms_pct"],
        "session_pass_rate_1000ms_pct":
            r["session_pass_rate_1000ms_pct"],
        "meets_95pct_requirement":
            r["event_pass_rate_1000ms_pct"] >= PASS_RATE_REQUIRED,
    })

for _, r in timer_threshold.iterrows():
    threshold_rows.append({
        "scenario": "timer_based",
        "setting": f"scheduler={int(r['scheduler_interval_ms'])}",
        "samples": int(r["samples"]),
        "p95_ms": r["p95_ms"],
        "passed_1000ms": int(r["passed_1000ms"]),
        "pass_rate_1000ms_pct": r["pass_rate_1000ms_pct"],
        "pass_rate_150ms_pct": r["pass_rate_150ms_pct"],
        "meets_95pct_requirement":
            r["pass_rate_1000ms_pct"] >= PASS_RATE_REQUIRED,
    })

threshold_summary = pd.DataFrame(threshold_rows)

threshold_summary.to_csv(
    DATA_DIR / "threshold_compliance_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# TẠO MARKDOWN SNIPPET TRỰC TIẾP TỪ CÁC DATAFRAME
#
# File này giúp kiểm tra rằng số trong phần viết Markdown
# được lấy từ đúng kết quả mà code vừa tính.
# ============================================================

lines = [
    "# Kết quả threshold được sinh tự động",
    "",
    "## Event-driven một phiên",
    "",
    "| Poller (ms) | P95 (ms) | Đạt ≤1s | Tỷ lệ | Kết luận |",
    "|---:|---:|---:|---:|---|",
]

for _, r in event_threshold.iterrows():
    status = (
        "Đáp ứng"
        if r["pass_rate_1000ms_pct"] >= PASS_RATE_REQUIRED
        else "Không đáp ứng"
    )

    lines.append(
        f"| {int(r['poller_interval_ms'])} "
        f"| {r['p95_ms']:.2f} "
        f"| {int(r['passed_1000ms'])}/{int(r['samples'])} "
        f"| {r['pass_rate_1000ms_pct']:.0f}% "
        f"| {status} |"
    )

lines += [
    "",
    "## Event-driven nhiều phiên",
    "",
    "| K | Event đạt ≤1s | Tỷ lệ event | Tỷ lệ từng phiên | Kết luận |",
    "|---:|---:|---:|---:|---|",
]

for _, r in multi_threshold.iterrows():
    status = (
        "Đáp ứng"
        if r["event_pass_rate_1000ms_pct"] >= PASS_RATE_REQUIRED
        else "Không đáp ứng"
    )

    lines.append(
        f"| {int(r['requested_session_count'])} "
        f"| {int(r['passed_events_1000ms'])}/{int(r['events'])} "
        f"| {r['event_pass_rate_1000ms_pct']:.0f}% "
        f"| {r['session_pass_rate_1000ms_pct']:.1f}% "
        f"| {status} |"
    )

lines += [
    "",
    "## Timer-based",
    "",
    "| Scheduler (ms) | P95 (ms) | Đạt ≤1s | Tỷ lệ | Kết luận |",
    "|---:|---:|---:|---:|---|",
]

for _, r in timer_threshold.iterrows():
    status = (
        "Đáp ứng"
        if r["pass_rate_1000ms_pct"] >= PASS_RATE_REQUIRED
        else "Không đáp ứng"
    )

    lines.append(
        f"| {int(r['scheduler_interval_ms'])} "
        f"| {r['p95_ms']:.2f} "
        f"| {int(r['passed_1000ms'])}/{int(r['samples'])} "
        f"| {r['pass_rate_1000ms_pct']:.0f}% "
        f"| {status} |"
    )

(DATA_DIR / "threshold_analysis_generated.md").write_text(
    "\n".join(lines),
    encoding="utf-8",
)

print()
print("=" * 72)
print("ĐÃ LƯU CÁC FILE SỐ LIỆU")
print("=" * 72)

for name in [
    "kb5_aggregated.csv",
    "kb6_aggregated.csv",
    "threshold_event_driven.csv",
    "threshold_multi_session.csv",
    "threshold_timer_based.csv",
    "threshold_compliance_summary.csv",
    "threshold_analysis_generated.md",
]:
    print(" -", DATA_DIR / name)



# ============================================================
# KB5 / KB6 THRESHOLD EXTENSION V3
# ============================================================

# Kịch bản 5:
# - total_core_ms dùng để kiểm tra riêng phần UCON Core.
# - event_end_to_end_latency_ms dùng để kiểm tra khi toàn bộ event
#   đã được backend xử lý xong.
# Lưu ý: KB5 không dùng event_end_to_end_latency_ms thay cho
# revoke_latency_ms vì đây không phải cùng một mốc thời gian.

kb5_threshold_rows = []

for k, group in kb5.groupby("requested_session_count"):
    kb5_threshold_rows.append({
        "K": int(k),
        "samples": len(group),

        "total_core_mean_ms":
            group["total_core_ms"].mean(),

        "total_core_p95_ms":
            group["total_core_ms"].quantile(0.95),

        "core_passed_1s":
            int((group["total_core_ms"] <= MAIN_THRESHOLD_MS).sum()),

        "core_pass_pct":
            100 * (group["total_core_ms"] <= MAIN_THRESHOLD_MS).mean(),

        "e2e_mean_ms":
            group["event_end_to_end_latency_ms"].mean(),

        "e2e_p95_ms":
            group["event_end_to_end_latency_ms"].quantile(0.95),

        "e2e_passed_1s":
            int(
                (
                    group["event_end_to_end_latency_ms"]
                    <= MAIN_THRESHOLD_MS
                ).sum()
            ),

        "e2e_pass_pct":
            100 * (
                group["event_end_to_end_latency_ms"]
                <= MAIN_THRESHOLD_MS
            ).mean(),
    })

kb5_threshold = (
    pd.DataFrame(kb5_threshold_rows)
    .sort_values("K")
)

kb5_threshold.to_csv(
    DATA_DIR / "threshold_kb5_core_vs_e2e.csv",
    index=False,
    encoding="utf-8-sig",
)


# Kịch bản 6:
# last_revoke_latency_ms là thời gian đến khi phiên mục tiêu
# bị thu hồi cuối cùng chuyển sang REVOKED.

kb6_threshold_rows = []

for n, group in kb6.groupby("total_active_sessions"):
    kb6_threshold_rows.append({
        "N": int(n),
        "K": 10,
        "samples": len(group),

        "last_revoke_mean_ms":
            group["last_revoke_latency_ms"].mean(),

        "last_revoke_p95_ms":
            group["last_revoke_latency_ms"].quantile(0.95),

        "last_revoke_passed_1s":
            int(
                (
                    group["last_revoke_latency_ms"]
                    <= MAIN_THRESHOLD_MS
                ).sum()
            ),

        "last_revoke_pass_pct":
            100 * (
                group["last_revoke_latency_ms"]
                <= MAIN_THRESHOLD_MS
            ).mean(),

        "total_core_mean_ms":
            group["total_core_ms"].mean(),

        "total_core_p95_ms":
            group["total_core_ms"].quantile(0.95),
    })

kb6_threshold = (
    pd.DataFrame(kb6_threshold_rows)
    .sort_values("N")
)

kb6_threshold.to_csv(
    DATA_DIR / "threshold_kb6_last_revoke.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# HÌNH 21 — KB5: Core vs E2E so với ngưỡng 1 giây
# ============================================================

x = np.arange(len(kb5_threshold))
width = 0.34

fig, ax = plt.subplots(figsize=(10.8, 6.2))

bars1 = ax.bar(
    x - width / 2,
    kb5_threshold["core_pass_pct"],
    width,
    edgecolor="white",
    linewidth=0.8,
    label="total_core_ms ≤ 1 giây",
)

bars2 = ax.bar(
    x + width / 2,
    kb5_threshold["e2e_pass_pct"],
    width,
    edgecolor="white",
    linewidth=0.8,
    label="E2E xử lý event ≤ 1 giây",
)

ax.axhline(
    PASS_RATE_REQUIRED,
    linestyle="--",
    linewidth=2,
    label="Tiêu chí chính: ≥95%",
)

for bars in [bars1, bars2]:
    for bar in bars:
        value = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 1.3,
            f"{value:.0f}%",
            ha="center",
            fontsize=9,
            
        )

ax.set_xticks(x)
ax.set_xticklabels(
    [f"K = {int(k)}" for k in kb5_threshold["K"]]
)

ax.set_ylim(0, 108)

ax.set_title(
    "Thời gian xử lý của UCON Core và toàn bộ event",
    fontsize=15,
)

ax.set_xlabel("Số phiên bị ảnh hưởng K")
ax.set_ylabel("Tỷ lệ mẫu không vượt quá 1 giây (%)")
ax.grid(axis="y", alpha=0.22)
ax.legend()

save(
    fig,
    "21_kb5_threshold_core_vs_e2e.png",
)


# ============================================================
# HÌNH 22 — KB6: last revoke so với ngưỡng 1 giây
# ============================================================

fig, ax = plt.subplots(figsize=(10.3, 6.1))

labels = [
    f"N = {int(n)}"
    for n in kb6_threshold["N"]
]

rates = kb6_threshold[
    "last_revoke_pass_pct"
].to_numpy()

bars = ax.bar(
    labels,
    rates,
    edgecolor="white",
    linewidth=0.8,
)

ax.axhline(
    PASS_RATE_REQUIRED,
    linestyle="--",
    linewidth=2,
    label="Tiêu chí chính: ≥95%",
)

for bar, value in zip(bars, rates):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 1.5,
        f"{value:.0f}%",
        ha="center",
        fontsize=10,
        
    )

ax.set_ylim(0, 105)

ax.set_title(
    "Kịch bản 6: Tỷ lệ đáp ứng ngưỡng 1 giây khi N tăng",
    fontsize=15,
    
)

ax.set_xlabel(
    "Tổng số phiên ACTIVE N (K = 10 cố định)"
)

ax.set_ylabel(
    "Tỷ lệ event có phiên cuối cùng REVOKED ≤ 1 giây (%)"
)

ax.grid(axis="y", alpha=0.22)
ax.legend()

save(
    fig,
    "22_kb6_threshold_last_revoke.png",
)


print()
print("=" * 72)
print("KB5 THRESHOLD")
print("=" * 72)
print(kb5_threshold.round(3))

print()
print("=" * 72)
print("KB6 THRESHOLD")
print("=" * 72)
print(kb6_threshold.round(3))


Đã tạo xong 5 biểu đồ:
 - d:\Work\Study\Learning\KLTN\Ucon_ABC\measurement-scripts\ucon_measurement_analysis_postupdate\charts\13_kb5_e2e_breakdown_with_poller.png
 - d:\Work\Study\Learning\KLTN\Ucon_ABC\measurement-scripts\ucon_measurement_analysis_postupdate\charts\14_kb5_core_breakdown.png
 - d:\Work\Study\Learning\KLTN\Ucon_ABC\measurement-scripts\ucon_measurement_analysis_postupdate\charts\15_kb5_core_component_share.png
 - d:\Work\Study\Learning\KLTN\Ucon_ABC\measurement-scripts\ucon_measurement_analysis_postupdate\charts\16_kb6_dependency_lookup.png
 - d:\Work\Study\Learning\KLTN\Ucon_ABC\measurement-scripts\ucon_measurement_analysis_postupdate\charts\17_kb6_total_core.png

ĐÁNH GIÁ NGƯỠNG 1 GIÂY

Event-driven, một phiên:
   poller_interval_ms  samples    p95_ms  passed_1000ms  pass_rate_1000ms_pct  \
0                 250      100   249.477            100                 100.0   
1                 500      100   497.206            100                 100.0   
2                1